# CIFAKE — Exploratory Data Analysis

**Goal**: Understand the CIFAKE dataset before training.
- Class distribution
- Sample image grids (Real vs Fake)
- Pixel-level statistics
- Frequency domain analysis (FFT) — reveals generator artifacts
- Mean image comparison

In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import cv2

# Add project root to path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"
print("Data directory:", DATA_DIR)

plt.style.use('dark_background')
PALETTE = {'REAL': '#34d399', 'FAKE': '#f87171'}

## 1. Dataset Statistics

In [ ]:
# Count images per split and class
stats = {}
for split in ['train', 'test']:
    for cls in ['REAL', 'FAKE']:
        folder = DATA_DIR / split / cls
        n = len(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
        stats[f"{split}/{cls}"] = n

df_stats = pd.DataFrame(list(stats.items()), columns=['Split/Class', 'Count'])
print(df_stats.to_string(index=False))
print(f"\nTotal images: {df_stats['Count'].sum():,}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE['REAL'] if 'REAL' in k else PALETTE['FAKE'] for k in stats.keys()]
ax.bar(stats.keys(), stats.values(), color=colors, edgecolor='none', alpha=0.85)
ax.set_title('Image Count per Split & Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(ax.patches, stats.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()

## 2. Sample Image Grid — Real vs Fake

In [ ]:
def load_samples(split, cls, n=16, seed=42):
    folder = DATA_DIR / split / cls
    paths = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
    random.seed(seed)
    return random.sample(paths, min(n, len(paths)))

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Sample Images — REAL (top 2 rows) vs FAKE (bottom 2 rows)',
             fontsize=14, fontweight='bold', y=1.01)

real_samples = load_samples('train', 'REAL', 16)
fake_samples = load_samples('train', 'FAKE', 16)

for i, path in enumerate(real_samples):
    row, col = divmod(i, 8)
    img = Image.open(path).resize((64, 64))
    axes[row, col].imshow(img)
    axes[row, col].axis('off')
    if col == 0:
        axes[row, col].set_ylabel('REAL', color=PALETTE['REAL'], fontsize=10, rotation=90)

for i, path in enumerate(fake_samples):
    row, col = divmod(i, 8)
    img = Image.open(path).resize((64, 64))
    axes[row+2, col].imshow(img)
    axes[row+2, col].axis('off')
    if col == 0:
        axes[row+2, col].set_ylabel('FAKE', color=PALETTE['FAKE'], fontsize=10, rotation=90)

plt.tight_layout(); plt.savefig('../outputs/eda_sample_grid.png', dpi=120, bbox_inches='tight'); plt.show()
print('Saved → outputs/eda_sample_grid.png')

## 3. Pixel Intensity Distribution

In [ ]:
def pixel_histogram(paths, n_imgs=200):
    all_pixels = []
    for p in paths[:n_imgs]:
        img = np.array(Image.open(p).convert('RGB').resize((32, 32))).flatten() / 255.
        all_pixels.extend(img)
    return np.array(all_pixels)

real_paths = list((DATA_DIR / 'train' / 'REAL').glob('*.jpg'))[:500]
fake_paths = list((DATA_DIR / 'train' / 'FAKE').glob('*.jpg'))[:500]

real_pix = pixel_histogram(real_paths)
fake_pix = pixel_histogram(fake_paths)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(real_pix, bins=64, alpha=0.65, color=PALETTE['REAL'], label='REAL', density=True)
ax.hist(fake_pix, bins=64, alpha=0.65, color=PALETTE['FAKE'], label='FAKE', density=True)
ax.set_title('Pixel Intensity Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Pixel Value (normalised)'); ax.set_ylabel('Density')
ax.legend()
plt.tight_layout(); plt.savefig('../outputs/eda_pixel_dist.png', dpi=120); plt.show()

## 4. Frequency Domain Analysis (FFT)

AI-generated images often show characteristic patterns in frequency space due to upsampling/convolutional artifacts.

In [ ]:
def mean_fft(paths, n_imgs=100, size=128):
    accum = np.zeros((size, size))
    for p in paths[:n_imgs]:
        img  = np.array(Image.open(p).convert('L').resize((size, size)), dtype=np.float32)
        f    = np.fft.fft2(img)
        fshift = np.fft.fftshift(f)
        mag  = np.log1p(np.abs(fshift))
        accum += mag
    return accum / n_imgs

real_fft = mean_fft(real_paths)
fake_fft = mean_fft(fake_paths)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(real_fft, cmap='viridis'); axes[0].set_title('Mean FFT — REAL', fontsize=12); axes[0].axis('off')
axes[1].imshow(fake_fft, cmap='viridis'); axes[1].set_title('Mean FFT — FAKE', fontsize=12); axes[1].axis('off')
diff = fake_fft - real_fft
im = axes[2].imshow(diff, cmap='RdYlGn'); axes[2].set_title('Difference (FAKE − REAL)', fontsize=12); axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)
fig.suptitle('Frequency Domain Analysis — AI artifacts visible in FFT', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('../outputs/eda_fft.png', dpi=120, bbox_inches='tight'); plt.show()

## 5. Mean Image Comparison

In [ ]:
def mean_image(paths, n_imgs=500, size=64):
    accum = np.zeros((size, size, 3), dtype=np.float64)
    for p in paths[:n_imgs]:
        img = np.array(Image.open(p).convert('RGB').resize((size, size)), dtype=np.float64)
        accum += img
    return (accum / n_imgs).astype(np.uint8)

real_mean = mean_image(real_paths)
fake_mean = mean_image(fake_paths)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(real_mean); axes[0].set_title('Mean REAL Image', fontsize=12, color=PALETTE['REAL']); axes[0].axis('off')
axes[1].imshow(fake_mean); axes[1].set_title('Mean FAKE Image', fontsize=12, color=PALETTE['FAKE']); axes[1].axis('off')
plt.tight_layout(); plt.savefig('../outputs/eda_mean_images.png', dpi=120, bbox_inches='tight'); plt.show()

## 6. Summary

Key observations from EDA:
- Dataset is **balanced** (equal REAL / FAKE counts)
- Pixel distributions differ slightly — FAKE images tend toward smoother tonal distributions
- **FFT analysis** reveals grid-like frequency artifacts in generated images (characteristic of convolution-based generators)
- Mean images look similar to the naked eye — classification requires learned features